# Live mapping with `for_leafmap`

When you bind a live `leafmap.maplibregl.Map` to a GeoAgent, the **mapping subagent** is added to the active subagent list and its tools (add layer, remove layer, set basemap, fly to, save) are wired to the very `Map` widget you passed in. Subsequent natural-language chats mutate that `Map` in place, so re-displaying the same widget shows the agent's work.


## Why `for_leafmap(m)` and not `chat(target_map=m)`?

Both paths bind the agent to a map, but they have different state semantics:

- `for_leafmap(m)` — builds the agent once with `map_obj=m` already in the context. The deepagents graph and its `MemorySaver` thread persist across many chats, so the agent can reason about what it did before.
- `agent.chat(query, target_map=m)` — switches an existing agent to a new map. The graph is rebuilt and a fresh thread id is rotated in ([`agent.py`](https://github.com/giswqs/geoagent/blob/main/geoagent/core/agent.py)), which resets conversation history. Use this when you genuinely want to hand the agent a different map mid-session.


## Setup

In [ ]:
from geoagent import for_leafmap
from leafmap.maplibregl import Map

In [ ]:
m = Map(center=[-83.92, 35.96], zoom=10)  # Knoxville, TN
agent = for_leafmap(m)
m

## Chat 1 — list current layers

In [ ]:
result = agent.chat("List the current map layers.")
print("Tools fired:", result.executed_tools)
if result.answer_text:
    print(result.answer_text[:400])

## Chat 2 — add a Sentinel-2 RGB COG

In [ ]:
result = agent.chat("Add a Sentinel-2 RGB COG layer over Knoxville TN for July 2024.")
print("Tools fired:", result.executed_tools)
m

## Chat 3 — change the basemap

In [ ]:
result = agent.chat("Switch the basemap to Carto Voyager.")
print("Tools fired:", result.executed_tools)
m

## Chat 4 — remove a layer

Because `for_leafmap` keeps a single thread alive, the agent remembers the layer names from chat 2 and can act on them by ordinal reference.

In [ ]:
result = agent.chat("Remove the Sentinel-2 layer.")
print("Tools fired:", result.executed_tools)
m

## Inspecting the mapping subagent's footprint

`result.executed_tools` enumerates every tool the agent actually invoked during the call, in order. For mapping queries you should see entries like `list_layers`, `add_layer`, `set_basemap`, and `remove_layer` from the mapping subagent.

In [ ]:
for query in (
    "Save the current map to /tmp/knoxville.html",
    "Zoom out to fit all layers.",
):
    r = agent.chat(query)
    print(f"{query!r:60s} -> {r.executed_tools}")